In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, RoFormerConfig, RoFormerModel
from torch.optim import AdamW
from datasets import load_dataset, load_from_disk
from tqdm import tqdm
from sklearn.metrics import f1_score
import os

b:\stacks\Machine_learning_projects\Video_Summarization_Translation\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Model Configuration (59M Parameters as per paper) [cite: 218]
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

class MiniSeg(nn.Module):
    def __init__(self, sentence_model_name=model_name):
        super(MiniSeg, self).__init__()
        self.sentence_encoder = AutoModel.from_pretrained(sentence_model_name)
        config = RoFormerConfig(
            num_hidden_layers=12, num_attention_heads=8,
            hidden_size=384, intermediate_size=1536, max_position_embeddings=2048
        )
        self.document_encoder = RoFormerModel(config)
        self.classifier = nn.Linear(384, 1)

    def forward(self, input_ids, attention_mask, grad_sample=0.5):
        # Gradient Sampling: Only backprop through sentence encoder for 50% of docs 
        use_grads = (torch.rand(1).item() < grad_sample) if self.training else True
        
        with torch.set_grad_enabled(use_grads):
            outputs = self.sentence_encoder(input_ids=input_ids, attention_mask=attention_mask)
            sentence_embeddings = outputs.last_hidden_state.mean(dim=1).unsqueeze(0) 
            
        doc_outputs = self.document_encoder(inputs_embeds=sentence_embeddings)
        return self.classifier(doc_outputs.last_hidden_state).squeeze(-1)

# 2. Optimized Dataset Wrapper (Moves tokenization to CPU workers)
class YTSEGDataset(Dataset):
    def __init__(self, hf_dataset, max_tokens=64, max_sentences=128):
        self.dataset = hf_dataset
        self.max_tokens = max_tokens
        self.max_sentences = max_sentences

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        sentences = item['text_ref'][:self.max_sentences]
        
        # Tokenize on CPU workers to keep GPU fed
        inputs = tokenizer(sentences, padding='max_length', truncation=True, 
                           return_tensors="pt", max_length=self.max_tokens)
        
        label_str = item['target_binary_ref'].replace("|=", "")
        labels = torch.tensor([int(c) for c in label_str[:self.max_sentences]]).float()
        
        return {
            'input_ids': inputs['input_ids'], 
            'attention_mask': inputs['attention_mask'], 
            'labels': labels
        }

def train_fast():
    # 3. Training Setup
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = MiniSeg().to(device)
    optimizer = AdamW(model.parameters(), lr=2.5e-5) # Paper LR 
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([10.0]).to(device)) # WBCE [cite: 143, 513]
    
    # 4. Faster Data Loading
    raw_dataset = load_from_disk("B:\\stacks\\Machine_learning_projects\\Video_Summarization_Translation\\datasets\\ytseg-dataset")
    train_set = YTSEGDataset(raw_dataset['train'].select(range(500)))
    val_set = YTSEGDataset(raw_dataset['validation'].select(range(100)))

    # Use num_workers > 0 and pin_memory for speed
    train_loader = DataLoader(train_set, batch_size=1, shuffle=True, 
                              num_workers=0, pin_memory=True) 
    val_loader = DataLoader(val_set, batch_size=1, shuffle=False, num_workers=2)

    # Mixed Precision & Accumulation
    scaler = torch.amp.GradScaler('cuda')
    accumulation_steps = 14  # To reach ~115k token effective batch size 

    for epoch in range(15): # 15 Epochs 
        model.train()
        optimizer.zero_grad()
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]")
        
        for i, batch in enumerate(train_pbar):
            ids = batch['input_ids'].squeeze(0).to(device, non_blocking=True)
            mask = batch['attention_mask'].squeeze(0).to(device, non_blocking=True)
            labels = batch['labels'].to(device, non_blocking=True)
            
            # AMP Autocast for speed and memory
            with torch.amp.autocast('cuda'):
                logits = model(ids, mask)
                min_len = min(logits.shape[1], labels.shape[1])
                loss = criterion(logits[:, :min_len], labels[:, :min_len])
                loss = loss / accumulation_steps # Normalize for accumulation

            scaler.scale(loss).backward()

            if (i + 1) % accumulation_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                
            train_pbar.set_postfix({'loss': f"{loss.item()*accumulation_steps:.4f}"})

        # 5. Validation Loop
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad(), torch.amp.autocast('cuda'):
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]"):
                ids = batch['input_ids'].squeeze(0).to(device)
                mask = batch['attention_mask'].squeeze(0).to(device)
                labels = batch['labels'].squeeze(0)
                
                logits = model(ids, mask).squeeze(0)
                preds = (torch.sigmoid(logits) > 0.5).int()
                
                min_len = min(len(preds), len(labels))
                all_preds.extend(preds[:min_len].cpu().numpy())
                all_labels.extend(labels[:min_len].cpu().numpy())

        print(f"Epoch {epoch+1} F1: {f1_score(all_labels, all_preds):.4f}")

if __name__ == "__main__":
    train_fast()

Epoch 1 [Val]:   0%|          | 0/100 [00:05<?, ?it/s]


RuntimeError: DataLoader worker (pid(s) 28652, 14224) exited unexpectedly